# Training the CNN model

In [ ]:
# Cargar split oficial de Samuel
import sys
from pathlib import Path
ROOT = Path(__file__).resolve().parents[2] if '__file__' in globals() else Path.cwd()
while not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data.preprocess import load_dataset_bundle
from src.utils.config import SPLIT_DIR, ROOT as CONFIG_ROOT

df_train, df_val, df_test, meta = load_dataset_bundle(SPLIT_DIR, CONFIG_ROOT)
print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
print("Class weights:", meta["class_weight_by_class"])

In [ ]:
# Configuración y modelo
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from src.utils.config import SEED, CLASSES, CNN_IMG_SIZE, MODELS_DIR, RESULTS_DIR
from src.utils.evaluation import save_predictions
from src.models.cnn import build_cnn_model

tf.keras.utils.set_random_seed(SEED)
IMG_SIZE = CNN_IMG_SIZE
BATCH = 32

# Pipeline tf.data
def make_ds(frame, shuffle):
    def load(path, y):
        img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        return tf.image.resize(img, IMG_SIZE), y
    ds = tf.data.Dataset.from_tensor_slices((frame.path.values, frame.label.values))
    if shuffle:
        ds = ds.shuffle(len(frame), seed=SEED)
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

ds_train = make_ds(df_train, shuffle=True)
ds_val = make_ds(df_val, shuffle=False)
ds_test = make_ds(df_test, shuffle=False)

# Modelo con augment=False y CNN_IMG_SIZE de config
model = build_cnn_model(input_shape=(*IMG_SIZE, 3), num_classes=len(CLASSES), augment=False)
model.summary()

# Entrenamiento con class_weight del metadata
from sklearn.utils.class_weight import compute_class_weight

class_weight = meta["class_weight_by_idx"]  # ya calculado por Samuel sobre train

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint(MODELS_DIR / "cnn_best.keras", monitor="val_loss", save_best_only=True),
]
history = model.fit(ds_train, validation_data=ds_val, epochs=25,
                    class_weight=class_weight, callbacks=callbacks)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(history.history).to_csv(RESULTS_DIR / "cnn_history.csv", index=False)

# Predicciones validación + test
for split_name, ds, df in [("val", ds_val, df_val), ("test", ds_test, df_test)]:
    proba = model.predict(ds, verbose=0)
    y_pred = [CLASSES[i] for i in proba.argmax(axis=1)]
    y_true = [CLASSES[i] for i in df.label]
    save_predictions(df.path.values, y_true, y_pred, proba, model_name="cnn", split=split_name)